In [1]:
from google.colab import files

# Kaggle API anahtarınızı yükleyin (kaggle.json dosyası)
files.upload()

# Anahtarı doğru dizine taşıyın ve izinleri ayarlayın
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
# Dataseti indirin (URL: https://www.kaggle.com/datasets/bhautikmangukiya12/hospital-inpatient-discharges-dataset)
!kaggle datasets download -d bhautikmangukiya12/hospital-inpatient-discharges-dataset

# Zip dosyasını açın
!unzip hospital-inpatient-discharges-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/bhautikmangukiya12/hospital-inpatient-discharges-dataset
License(s): CC0-1.0
404 - Not Found - No gcs url found
unzip:  cannot find or open hospital-inpatient-discharges-dataset.zip, hospital-inpatient-discharges-dataset.zip.zip or hospital-inpatient-discharges-dataset.zip.ZIP.


In [4]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhautikmangukiya12/hospital-inpatient-discharges-dataset")

print("Path to dataset files:", path)

KaggleApiHTTPError: 404 Client Error.

Resource not found at URL: https://www.kaggle.com/datasets/bhautikmangukiya12/hospital-inpatient-discharges-dataset/versions/1
The server reported the following issues: No gcs url found
Please make sure you specified the correct resource identifiers.

In [3]:
!ls  # hospital_discharge_data.csv dosyasını göreceksiniz

sample_data


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, classification_report,accuracy_score

In [ ]:
df = pd.read_csv('hospital_discharge_data.csv')
df.head(50)

In [ ]:
df.info()

In [ ]:
describe = df.describe()
describe

In [ ]:
los = df['Length of Stay']
df['Length of Stay'] = df['Length of Stay'].replace("120 +",120)
df['Length of Stay'] = pd.to_numeric(df['Length of Stay'])
los = df['Length of Stay']

In [ ]:
df.isna().sum()

In [ ]:
for column in df.columns:
  unique_values =len(df[column].unique())
  print(f"Number of unique values in {column}: {unique_values}")

In [ ]:
df = df[df["Patient Disposition"] != "Expired"]

In [ ]:
sns.boxplot(x="Payment Typology 1", y= "Length of Stay",data =df)
plt.title("Payment Typology 1 vs Length of Stay")
plt.xticks(rotation=60)
plt.show()

In [ ]:
sns.countplot(x = "Age Group",data = df[df["Payment Typology"]] =="Medicare")
plt.title("Medicare Partients for Age Group")
plt.show()

In [ ]:
sns.boxplot(x="Type pf Admission", y= "Length of Stay",data =df)
plt.title("Type pf Admission vs Length of Stay")
plt.xticks(rotation=60)
plt.show()

In [ ]:
f,ax = plt.subplots()
sns.boxplot(x="Age Group", y= "Length of Stay",data =df)
plt.title("Age Group vs Length of Stay")
plt.xticks(rotation=60)
ax.set(ylim=(0,25))
plt.show()

In [ ]:
df = df.drop(["Hospital Service Area","Hospital County","Operating Certificate Number",
              "Facility Name","Zip Code - 3 digits","Patient Disposition","Discharge Year",
              "CCSR Diagnosis Description","CCSR Procedure Description","APR DRG Description",
              "APR MDC Description","ADR Severity of Illness Description",
              "Payment Typology 2","Payment Typology 3","Birth Weight","Total Charge","Total Cost"],axis=1)

In [ ]:
age_group_index = {"0 to 17":1,"18 to 29":2,"30 to 49":3,"50 to 69":4,"70 or Older":5}
gender_index = {"U":0,"F":1,"M":2}
risk_and_severity_index = {np.nan:0,"Minor":1,"Moderate":2,"Major":3,"Extreme":4}


In [ ]:
df["Age Group"] = df["Age Group"].apply(lambda x: age_group_index[x])
df["Gender"] = df["Gender"].apply(lambda x: gender_index[x])
df["APR Severity of Illness Description"] = df["APR Severity of Illness Description"].apply(lambda x: risk_and_severity_index[x])
df

In [ ]:
encoder = OrdinalEncoder()
df["Race"] = encoder.fit_transform(df[["Race"]].reshape(-1,1))
df["Ethnicity"] = encoder.fit_transform(df[["Ethnicity"]].reshape(-1,1))
df["Type of Admission"] = encoder.fit_transform(df[["Type of Admission"]].reshape(-1,1))
df["CCSR Diagnosis Code"] = encoder.fit_transform(df[["CCSR Diagnosis Code"]].reshape(-1,1))
df["CCSR Procedure Code"] = encoder.fit_transform(df[["CCSR Procedure Code"]].reshape(-1,1))
df["APR Medical Surgical Description"] = encoder.fit_transform(df[["APR Medical Surgical Description"]].reshape(-1,1))
df["Payment Typology"] = encoder.fit_transform(df[["Payment Typology"]].reshape(-1,1))
df["Emergency Department Indicator"] = encoder.fit_transform(df[["Emergency Department Indicator"]].reshape(-1,1))

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop("CCSR Procedure Code",axis=1)
df = df.dropna(subset=["Permanent Facility Id","CCSR Diagnosis Code"])


In [ ]:
# train test split
X = df.drop("Length of Stay",axis=1)
y = df["Length of Stay"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [ ]:
# Regression: Train ve Test

dtree = DecisionTreeRegressor(max_depth=10)
dtree.fit(X_train,y_train)
train_prediction = dtree.predict(X_train)
test_prediction = dtree.predict(X_test)

In [ ]:
print("RMSE: Train: ",np.sqrt(mean_squared_error(y_train,train_prediction)))
print("RMSE: Test: ",np.sqrt(mean_squared_error(y_test,test_prediction)))

In [ ]:
bins = [0,5,10,20,30,50,120]
labels = [5,10,20,3,50,120]

In [ ]:
df["los_bin"] = pd.cut(df["Length of Stay"],bins=bins)
df["los_label"] = pd.cut(df["Length of Stay"],bins=bins,labels=labels)
df = dfçhead(50)
df["los_bin"] = df["los_bin"].apply(lambda x: str(x)).replace(","," -")
df["los_bin"] = df["los_bin"].apply(lambda x: str(x)).replace("120","120+")

In [ ]:
f,ax = plt.subplots()
sns.countplot(x="los_bin",data=df)

In [ ]:
new_X = df.drop(["Length of Stay","los_bin","los_label"],axis=1)
new_y = df["los_label"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(new_X,new_y,test_size=0.2,random_state=42)

In [ ]:
dtree =DecisionTreeClassifier(max_depth=10)
dtree.fit(X_train,y_train)

In [ ]:
train_prediction = dtree.predict(X_train)
test_prediction = dtree.predict(X_test)

In [ ]:
print("Train Accuracy: ",accuracy_score(y_train,train_prediction))
print("Test Accuracy: ",accuracy_score(y_test,test_prediction))
print("Classification Report: ",callassification_report(y_test,test_prediction))